In [2]:
import sys
sys.path.append(rf"/Users/baia/Desktop/PYTHON/mba_dsa_usp_esalq")

from TCC.utils.constantes import *
import matplotlib.pyplot as plt

## TOTAL3 (Altcoin Market Cap (Excluding ETH & Stablecoins, BTC)
- O valor de mercado total de todas as criptomoedas, subtraindo Bitcoin, Ethereum e Stablecoins. Muitas vezes representado pelo ticker TOTAL3 no TradingView.

- Representa o "Apetite por Risco Extremo". É o índice das Shitcoins, Memecoins e projetos menores ("Alts").

- Fonte: https://br.tradingview.com/symbols/TOTAL3/

### HIPÓTESES
- Hipótese Era Varejo: Crescimento explosivo dessa métrica era a causa direta da queda do $BTC.D$ (Altseason irracional). O crescimento do TOTAL3 era o driver da queda da Dominância. Era o dinheiro saindo do BTC para ICOs.

- Hipótese Era Institucional: Instituições monitoram isso como indicador de "Exuberância Irracional". Se o TOTAL3 sobe enquanto o S&P 500 cai, é uma divergência insustentável que geralmente acaba em correção severa (o que aumenta a dominância do BTC via Flight to Quality).

### TRATAMENTO
- O desvio padrão dos retornos (std) é de 4.9% ao dia. O máximo (max) é 48.9% em um único dia.
- Modelos como XGBoost adoram essa volatilidade, mas ela pode gerar ruído. O Log-Retorno é essencial para "domar" essa escala que foi de 600 milhões (2017) para 900 bilhões (2021).

In [13]:
df_total3 = (pd.read_csv(rf"raw/201501_mkcap_total3.csv")

                         .assign(Data_UTC = lambda df: pd.to_datetime(df['time'], unit='s', utc=True))
                         .assign(Data_UTC = lambda df: df['Data_UTC'].dt.strftime("%Y-%m-%d"))

                        .rename(columns={'close': 'close_price'}) 

                        [['Data_UTC', 'close_price']] 
 
)
df_total3

df_total3_log_Ret =(
    df_periodo
        .merge(df_total3, how='left', on='Data_UTC')
        .assign(close_price = lambda df: df['close_price'].ffill())
        .assign(Total3_Log_Ret = lambda df: np.log(df['close_price']) - np.log(df['close_price'].shift(1)))


        .query("Data_UTC > '2017-01-03'")
        [['Data_UTC','close_price','Total3_Log_Ret']]

)
df_total3_log_Ret

# print_dataframe_info(df_total3_log_Ret, "Total3 Market Cap")

,Data_UTC,close_price,Total3_Log_Ret
4,2017-01-04,8.000004e+08,0.079352
5,2017-01-05,7.574391e+08,-0.054669
6,2017-01-06,7.014694e+08,-0.076766
7,2017-01-07,6.906900e+08,-0.015486
8,2017-01-08,6.991268e+08,0.012141
...,...,...,...
3131,2025-07-28,7.721518e+11,-0.039505
3132,2025-07-29,7.661266e+11,-0.007834
3133,2025-07-30,7.543308e+11,-0.015516
3134,2025-07-31,7.336012e+11,-0.027866


## Stablecoin Supply Ratio (SSR)
- A razão entre o Market Cap do Bitcoin e o Market Cap de todas as Stablecoins.
- Mede o "Poder de Compra Latente".
    - SSR Baixo: As stablecoins têm muito poder de compra em relação ao tamanho do BTC (Bullish).
    - SSR Alto: O Bitcoin está "caro" em relação à liquidez disponível (Bearish).

- Fonte: https://www.tradingview.com/script/4yIVcZS3-Stablecoin-Supply-Ratio-Oscillator/

### HIPÓTESES
- Era Varejo: O SSR era alto e menos relevante, pois a maior parte do par de negociação era BTC/USD (fiat) e não BTC/USDT.

- Era Institucional: Hipótese central. Hoje, a liquidez global de cripto é dominada por USDT e USDC.
- Na era institucional, as stablecoins (USDT/USDC) são usadas como "Caixa" e ferramenta de liquidação. Um SSR caindo indica que o mercado está acumulando "pólvora". Se essa pólvora for gasta em BTC, a Dominância sobe. Se for gasta em Alts, a Dominância cai.
    - SSR Caindo: Aumento da oferta de stablecoins (impressão de Tether) ou queda do preço do BTC. Ambos sinalizam acumulação potencial ("Dry Powder").
    - SSR Subindo: O BTC valorizou muito frente à liquidez disponível. Sinal de exaustão (Topo local).


### TRATAMENTO
- Primeira Diferença (após Forward Fill)
- Justificativa: Como um oscilador de liquidez, a diferença diária captura a injeção (ou retirada) líquida de poder de compra. Variações negativas indicam aumento relativo da "munição" dos compradores (bullish), enquanto variações positivas indicam esticamento da valorização (bearish).

In [24]:
df_ssr = (pd.read_csv(rf"raw/201404_ssr.csv")

                         .assign(Data_UTC = lambda df: pd.to_datetime(df['time'], utc=True))
                         .assign(Data_UTC = lambda df: df['Data_UTC'].dt.strftime("%Y-%m-%d"))

                        [['Data_UTC', 'SSR']] 
 
)
df_ssr

df_ssr_diff =(
    df_periodo
        .merge(df_ssr, how='left', on='Data_UTC')
        .assign(SSR = lambda df: df['SSR'].ffill())
        .assign(SSR_diff = lambda df: df['SSR'].diff())

        .query("Data_UTC > '2017-01-03'")
        [['Data_UTC','SSR_diff']]
)
df_ssr_diff

# print_dataframe_info(df_ssr_diff, "SSR Diff") 

,Data_UTC,SSR_diff
4,2017-01-04,0.117991
5,2017-01-05,-0.140475
6,2017-01-06,-0.112826
7,2017-01-07,0.015159
8,2017-01-08,-0.305836
...,...,...
3131,2025-07-28,-0.010777
3132,2025-07-29,-0.001299
3133,2025-07-30,-0.002737
3134,2025-07-31,-0.018982


## Flippening_Ratio (ETH_Cap/BTC_Cap)
- Diferença diária da razão ETH_Cap / BTC_Cap. Mede a velocidade com que o Ethereum ganha ou perde terreno frente ao Bitcoin, ajustado pela oferta (supply) de ambos.
- É o indicador definitivo de Rotação de Narrativa. O mercado cripto oscila entre "Ouro Digital" (BTC) e "Computador Mundial" (ETH).

- Fonte: https://br.tradingview.com/ (CRYPTOCAP:ETH/CRYPTOCAP:BTC)

### HIPÓTESES
- Hipótese (Era Varejo)Alta Volatilidade e Expectativa. Pré-2020 (especialmente 2017), o sonho do "Flippening" movia o mercado. Variações positivas nessa razão causavam quedas abruptas no $BTC.D$.

- Hipótese (Era Institucional)Estabilização. Pós-2020, o mercado entendeu que são ativos diferentes (Commodity vs Tech). Espera-se que a volatilidade desse índice tenha diminuído e que ele oscile dentro de bandas mais definidas, sem ameaçar a hegemonia do BTC tão frequentemente.


### TRATAMENTO
- Primeira Diferença (Diff)
- Justificativa: A série é uma razão de preços/oferta que oscila em bandas. A diferença diária captura o momento exato da rotação de narrativas. Valores positivos indicam outperformance do ETH (pressão baixista na BTC.D).

In [31]:
df_flippening = (pd.read_csv(rf"raw/201809_Flippening_Ratio_Diff.csv")

                         .assign(Data_UTC = lambda df: pd.to_datetime(df['time'], utc=True))
                         .assign(Data_UTC = lambda df: df['Data_UTC'].dt.strftime("%Y-%m-%d"))

                        .rename(columns={'close': 'close_price'}) 

                        [['Data_UTC', 'close_price']] 
 
)
df_flippening

df_flippening_diff =(
    df_periodo
        .merge(df_flippening, how='left', on='Data_UTC')
        .assign(close_price = lambda df: df['close_price'].ffill())
        .assign(close_price_diff = lambda df: df['close_price'].diff())
        .query("Data_UTC > '2020-01-01'")
        [['Data_UTC','close_price_diff']]
)
df_flippening_diff
# print_dataframe_info(df_flippening_diff, "Flippening Diff") 

,Data_UTC,close_price_diff
1097,2020-01-02,0.000675
1098,2020-01-03,0.000247
1099,2020-01-04,-0.000507
1100,2020-01-05,0.001115
1101,2020-01-06,0.000989
...,...,...
3131,2025-07-28,-0.001599
3132,2025-07-29,-0.000005
3133,2025-07-30,0.000999
3134,2025-07-31,-0.002337
